# hipinfer on Kaggle: 2x T4, tensor parallelism

LLM inference engine written from scratch in HIP, built here with `nvcc`. Kaggle's free **GPU T4 x2**
machine runs both halves of the benchmark: every model on one T4, then sharded over both T4s with
tensor parallelism (Megatron-style column/row-parallel layers, a custom one-shot all-reduce kernel for
decode-sized messages, NCCL for prefill-sized ones, prefill all-reduces overlapped with compute).
Llama-3.1-8B (16 GB of fp16 weights) does not fit one 15 GB T4 and runs only sharded.

**Settings -> Accelerator -> GPU T4 x2**, and **Internet on** (weights come from Hugging Face).
Run all. It takes ~2 h (most of it HF reference runs and exporting the 8B weights). Results land
in `/kaggle/working/hipinfer_results_kaggle_2xT4.zip` (Output tab).

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!nvidia-smi topo -m
!nvcc --version | tail -2
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())

Unpack the sources (embedded below so no clone is needed).

In [ ]:
import base64, io, tarfile, os
BLOB = "..."
os.makedirs("/kaggle/working/hipinfer", exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(BLOB)), mode="r:gz").extractall("/kaggle/working/hipinfer")
%cd /kaggle/working/hipinfer

Single-GPU suite (kernel tests, HF token gate, perplexity vs HF, benchmarks) for `MODELS`, then the
tensor-parallel stage for `TP_MODELS`: TP-vs-1-GPU token test, all-reduce latency, decode/prefill/batched
benchmarks against one GPU, and for the 8B (sharded only) perplexity vs HF spread over both GPUs. Weights
and the HF cache go to /tmp (the 20 GB /kaggle/working limit is too small for the 8B).

In [ ]:
!pip install -q datasets nvidia-ml-py
import os
os.environ["HF_HOME"] = "/tmp/hf"
MODELS = "tinyllama llama32-1b qwen25-1.5b llama32-3b"
TP_MODELS = "tinyllama llama32-1b qwen25-1.5b llama32-3b llama31-8b"
!chmod +x scripts/run_all.sh
!mkdir -p /tmp/w
!PY=python WEIGHTS_DIR=/tmp/w PPL_WINDOWS=8 TP=2 TP_MODELS="{TP_MODELS}" scripts/run_all.sh kaggle_2xT4 {MODELS}

In [ ]:
!python scripts/summarize.py results
!zip -qr /kaggle/working/hipinfer_results_kaggle_2xT4.zip results